# 11. LoRA ir Full Comparison

In [1]:
import json, math, pandas as pd

rows = []
for run in ["full", "lora"]:
    log = json.load(open(f"../outputs/{run}/log_history.json"))
    evals = [r for r in log if "eval_loss" in r]
    best = min(evals, key=lambda r: r["eval_loss"])
    final = [r for r in log if "train_runtime" in r][-1]
    rows.append({"method": run,
                 "best_val_loss": round(best["eval_loss"], 3),
                 "best_val_ppl": round(math.exp(best["eval_loss"]), 1),
                 "best_step": best["step"],
                 "train_time_min": round(final["train_runtime"] / 60, 1)})
cmp = pd.DataFrame(rows).set_index("method")

ev = pd.read_json("../outputs/eval_results.jsonl", lines=True).drop_duplicates(["model", "split"], keep="last")
ev["method"] = ev["model"].str.replace("outputs/", "").str.replace("/model", "")
cmp = cmp.join(ev.pivot_table(index="method", columns="split", values="perplexity")
                 .add_suffix("_ppl").round(1))
cmp["trainable_params"] = {"full": 82_318_080, "lora": 405_504}
cmp

,best_val_loss,best_val_ppl,best_step,train_time_min,test_ppl,trainable_params
method,,,,,,
full,3.918,50.3,2000,2.2,51.1,82318080
lora,4.016,55.5,2000,2.0,56.4,405504
